# Module 6: Pandas — Data Wrangling & Analysis

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
- [ ] Create Series and DataFrames from scratch and from files
- [ ] Select rows and columns using `loc` and `iloc`
- [ ] Handle missing data (detect, fill, or drop)
- [ ] Transform data using `apply` and built-in methods
- [ ] Group and summarise data with `groupby`
- [ ] Combine DataFrames using `merge` and `concat`

**⏱ Estimated Time:** 75–90 minutes  
**📋 Prerequisites:** Modules 1–5 (Python basics, NumPy)


## 📦 Environment Setup


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 10)  # Keeps output tidy
print(f"Pandas version: {pd.__version__}")
print("✅ Ready to go!")


In [ ]:
# Helper functions — run once
from IPython.display import HTML, display

def info_box(title, content):
    display(HTML(f"""
    <div style="background:#E3F2FD;padding:15px;border-left:5px solid #2196F3;
    margin:10px 0;border-radius:4px;">
    <strong>💡 {title}</strong><br>{content}</div>"""))

def warning_box(title, content):
    display(HTML(f"""
    <div style="background:#FFF3E0;padding:15px;border-left:5px solid #FF9800;
    margin:10px 0;border-radius:4px;">
    <strong>⚠️ {title}</strong><br>{content}</div>"""))

print("✅ Helpers loaded!")


---
## 1. Series and DataFrames — The Two Core Objects

**Pandas** is built on two objects:

| Object | What it is | Think of it as |
|--------|-----------|----------------|
| **Series** | A single column of data | A labelled list |
| **DataFrame** | A table of data | A spreadsheet or Excel sheet |

A DataFrame is really just a collection of Series (one per column).


In [ ]:
# --- Creating a Series ---
ages = pd.Series([25, 30, 35, 28], name="age")
print(ages)
print(f"\nType: {type(ages)}")


In [ ]:
# --- Creating a DataFrame from a dictionary ---
data = {
    "name":   ["Alice", "Bob", "Charlie", "Diana"],
    "age":    [25, 30, 35, 28],
    "city":   ["London", "Manchester", "London", "Edinburgh"],
    "salary": [45000, 52000, 61000, 48000]
}

df = pd.DataFrame(data)
df


In [ ]:
# Quick overview of your data
print(f"Shape: {df.shape}  (rows, columns)")
print(f"Columns: {list(df.columns)}")
print()
df.info()


---
## 2. Selecting Data — `loc` and `iloc`

There are two main ways to select data:
- **`loc`** — select by **label** (column name, row index label)
- **`iloc`** — select by **integer position** (0, 1, 2, ...)


In [ ]:
# Select a single column (returns a Series)
print(df["name"])


In [ ]:
# Select multiple columns (returns a DataFrame)
df[["name", "salary"]]


In [ ]:
# loc — select by label
# Row at index 0, columns "name" and "city"
print(df.loc[0, ["name", "city"]])

print()

# All rows where city is London
df.loc[df["city"] == "London"]


In [ ]:
# iloc — select by position
# First 2 rows, first 3 columns
df.iloc[:2, :3]


In [ ]:
info_box(
    "loc vs iloc — Quick Rule",
    "<b>loc</b> = <b>l</b>abel-based → uses column <b>names</b> and index <b>labels</b><br>"
    "<b>iloc</b> = <b>i</b>nteger-based → uses <b>numbers</b> (0, 1, 2, ...)<br>"
    "When in doubt, use <code>loc</code> — it's more readable."
)


---
## 3. Loading Data from Files

In real projects, you'll almost always load data from a file rather than typing it by hand.


In [ ]:
# Let's create a CSV file first, then load it
df.to_csv("employees.csv", index=False)
print("✅ CSV saved!")

# Now load it back
df_loaded = pd.read_csv("employees.csv")
df_loaded


In [ ]:
# Useful read_csv parameters:
# pd.read_csv("file.csv", sep=";")       # Different separator
# pd.read_csv("file.csv", nrows=100)     # Only first 100 rows
# pd.read_csv("file.csv", usecols=["name", "age"])  # Only some columns


---
## 4. Handling Missing Data

Real-world data is almost always messy. Some values will be missing. Pandas uses `NaN` (Not a Number) to represent missing values.


In [ ]:
# Create a DataFrame with missing values
messy = pd.DataFrame({
    "name":   ["Alice", "Bob", "Charlie", "Diana"],
    "age":    [25, np.nan, 35, 28],
    "score":  [88, 92, np.nan, np.nan]
})
messy


In [ ]:
# Detect missing values
print("Missing per column:")
print(messy.isna().sum())


In [ ]:
# Option 1: Drop rows with ANY missing value
print("Drop rows with NaN:")
print(messy.dropna())


In [ ]:
# Option 2: Fill missing values
filled = messy.copy()
filled["age"] = filled["age"].fillna(filled["age"].mean())       # Fill with mean
filled["score"] = filled["score"].fillna(0)                       # Fill with 0
filled


In [ ]:
warning_box(
    "Never Blindly Drop Missing Data",
    "Dropping rows can remove important information. Always ask: <b>Why</b> is this data missing? "
    "Sometimes filling with the mean or median is better. Sometimes the missingness itself is informative."
)


---
## 5. Transforming Data

Pandas makes it easy to create new columns, modify existing ones, and apply custom functions.


In [ ]:
# Add a new column based on a calculation
df["bonus"] = df["salary"] * 0.10
df


In [ ]:
# Use .apply() to run a custom function on every value in a column
def salary_band(salary):
    if salary >= 55000:
        return "Senior"
    elif salary >= 45000:
        return "Mid"
    else:
        return "Junior"

df["band"] = df["salary"].apply(salary_band)
df


In [ ]:
# Rename columns
df_renamed = df.rename(columns={"name": "employee_name", "city": "location"})
df_renamed.columns.tolist()


---
## 6. GroupBy — Split, Apply, Combine

`groupby` is one of the most powerful features in Pandas. It lets you:
1. **Split** the data into groups
2. **Apply** a function to each group
3. **Combine** the results back together

**Analogy:** It's like sorting students into class groups, calculating each group's average, and then reporting the results.


In [ ]:
# Average salary by city
df.groupby("city")["salary"].mean()


In [ ]:
# Multiple aggregations at once
df.groupby("city")["salary"].agg(["mean", "min", "max", "count"])


In [ ]:
# Group by band and get multiple stats
df.groupby("band").agg(
    avg_salary=("salary", "mean"),
    avg_age=("age", "mean"),
    count=("name", "count")
)


---
## 7. Combining DataFrames — merge and concat

Often your data lives in multiple tables and you need to combine them.

- **`merge`** — joins two DataFrames on a shared column (like a SQL JOIN)
- **`concat`** — stacks DataFrames on top of each other or side by side


In [ ]:
# Create two related DataFrames
employees = pd.DataFrame({
    "emp_id": [1, 2, 3, 4],
    "name":   ["Alice", "Bob", "Charlie", "Diana"]
})

departments = pd.DataFrame({
    "emp_id": [1, 2, 3, 5],
    "dept":   ["Engineering", "Marketing", "Engineering", "HR"]
})

# Inner merge — only matching emp_ids
pd.merge(employees, departments, on="emp_id", how="inner")


In [ ]:
# Left merge — keep ALL employees, even if no department match
pd.merge(employees, departments, on="emp_id", how="left")


In [ ]:
# Concat — stack DataFrames vertically
batch1 = pd.DataFrame({"name": ["Alice", "Bob"], "score": [85, 90]})
batch2 = pd.DataFrame({"name": ["Charlie"], "score": [78]})

combined = pd.concat([batch1, batch2], ignore_index=True)
combined


In [ ]:
info_box(
    "Merge Types (how=)",
    "<b>inner</b> — only rows that match in BOTH tables<br>"
    "<b>left</b> — all rows from the LEFT table, matches from right<br>"
    "<b>right</b> — all rows from the RIGHT table, matches from left<br>"
    "<b>outer</b> — all rows from BOTH tables"
)


---
## 🏋️ Practice Exercises


### Exercise 1: Explore a DataFrame ⭐

Create a DataFrame of 5 books with columns: `title`, `author`, `pages`, `rating`. Then:
1. Display only books with rating >= 4
2. Calculate the average number of pages


In [ ]:
# YOUR CODE HERE




### Exercise 2: Handle Missing Data ⭐⭐

Create this DataFrame and fill missing ages with the median age, then drop rows where score is missing.

```
name     age   score
Alice    25    88
Bob      NaN   92
Charlie  35    NaN
Diana    28    76
Eve      NaN   NaN
```


In [ ]:
# YOUR CODE HERE




### Exercise 3: GroupBy Analysis ⭐⭐

Using the DataFrame below, find the average score per department and the name of the top scorer in each department.

```python
data = {
    "name": ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "dept": ["Sales", "Engineering", "Sales", "Engineering", "Sales"],
    "score": [88, 95, 72, 91, 85]
}
```


In [ ]:
# YOUR CODE HERE
data = {
    "name": ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "dept": ["Sales", "Engineering", "Sales", "Engineering", "Sales"],
    "score": [88, 95, 72, 91, 85]
}




---
## 📋 Solutions

<details>
<summary>Click to reveal Exercise 1 solution</summary>

```python
books = pd.DataFrame({
    "title": ["Book A", "Book B", "Book C", "Book D", "Book E"],
    "author": ["Author 1", "Author 2", "Author 3", "Author 4", "Author 5"],
    "pages": [320, 150, 480, 220, 360],
    "rating": [4.5, 3.8, 4.2, 3.5, 4.8]
})
print(books[books["rating"] >= 4])
print(f"Average pages: {books['pages'].mean()}")
```

</details>

<details>
<summary>Click to reveal Exercise 2 solution</summary>

```python
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "age": [25, np.nan, 35, 28, np.nan],
    "score": [88, 92, np.nan, 76, np.nan]
})
df["age"] = df["age"].fillna(df["age"].median())
df = df.dropna(subset=["score"])
print(df)
```

</details>

<details>
<summary>Click to reveal Exercise 3 solution</summary>

```python
df = pd.DataFrame(data)
print(df.groupby("dept")["score"].mean())
print()
# Top scorer per department
idx = df.groupby("dept")["score"].idxmax()
print(df.loc[idx, ["dept", "name", "score"]])
```

</details>


---
## 🎯 Key Takeaways

1. **Series** = one column; **DataFrame** = a table (collection of Series).
2. Use **`loc`** (labels) and **`iloc`** (integer positions) to select data.
3. **Missing data** — detect with `.isna()`, handle with `.fillna()` or `.dropna()`.
4. **`.apply()`** runs a custom function on every value in a column.
5. **`groupby`** is your go-to for summarising data by category.
6. **`merge`** joins tables on shared columns; **`concat`** stacks them.

## ✅ Self-Assessment
- [ ] I can create a DataFrame and select specific rows/columns
- [ ] I can handle missing values appropriately
- [ ] I can use groupby to summarise data

## 📚 Next Steps
- **Next Module:** Module 7 — Data Visualisation
- **Practice:** Load a real CSV file (try Kaggle.com for free datasets) and explore it with these tools
